In [26]:
%pip install pandas
%pip install numpy
%pip install apyori
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import pandas as pd
import numpy as np
import pickle
from apyori import apriori
from mlxtend.preprocessing import TransactionEncoder
from datetime import datetime
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [28]:
# Parameters (to be inserted by Flask app)
file_path = '../uploads/data_kecelakaan.xlsx'

# Load the data
df = pd.read_excel(file_path, engine='openpyxl')

c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\packaging\core.py:99: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.datetime.utcnow()
c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\packaging\core.py:99: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.datetime.utcnow()


In [29]:
##tanggal dan waktu ## PERCOBAAN
import re

data_tanggal_waktu = df[['no.','waktu kejadian']]
# Loop melalui setiap baris dan ambil informasi hari dan jam
# Kamus untuk koreksi kesalahan penulisan
hari_koreksi = {
    "JUMAYT": "JUMAT",
    "SENIN": "SENIN",   # Tambahkan koreksi jika ada kesalahan lain
    "KAMIS": "KAMIS",
    "REBU": "RABU"
}
bulan_koreksi = {
    "JANAURI": "JANUARI",
    "AJANUARI": "JANUARI",
    "PEBRUARI": "FEBRUARI",
    "JANUAROI": "JANUARI",
    "PEBRUIARI" : "FEBRUARI"
}
def categorize_time(time_str):
    time_obj = datetime.strptime(time_str, '%H.%M').time()
    
    if time_obj >= datetime.strptime('00:00', '%H:%M').time() and time_obj <= datetime.strptime('03:59', '%H:%M').time():
        return 'Dini Hari'
    elif time_obj >= datetime.strptime('04:00', '%H:%M').time() and time_obj <= datetime.strptime('10:59', '%H:%M').time():
        return 'Pagi'
    elif time_obj >= datetime.strptime('11:00', '%H:%M').time() and time_obj <= datetime.strptime('14:59', '%H:%M').time():
        return 'Siang'
    elif time_obj >= datetime.strptime('15:00', '%H:%M').time() and time_obj <= datetime.strptime('17:59', '%H:%M').time():
        return 'Sore'
    elif time_obj >= datetime.strptime('18:00', '%H:%M').time() and time_obj <= datetime.strptime('18:59', '%H:%M').time():
        return 'Petang'
    elif time_obj >= datetime.strptime('19:00', '%H:%M').time() and time_obj <= datetime.strptime('23:59', '%H:%M').time():
        return 'Malam'
    else:
        return 'lainnya'
hari_list = []
bulan_list = []
waktu_list = []
data_tanggal_waktu_valid = []

for index, row in data_tanggal_waktu.iterrows():
    tanggal_waktu = row['waktu kejadian']
    tanggal_waktu = tanggal_waktu.replace("JAM", "").replace("WIB", "").strip()
    id = row['no.']
    match = re.match(r'(\w+),\s*(\d{2} \w+ \d{4}),\s*(\d{2}\.\d{2})', tanggal_waktu)
    if match:
        hari, bulan_tahun, waktu = match.groups()
        # Koreksi hari dan bulan jika ada kesalahan penulisan
        hari = hari_koreksi.get(hari, hari)
        bulan = bulan_koreksi.get(bulan_tahun.split()[1], bulan_tahun.split()[1])
        waktu = categorize_time(waktu)
        data_tanggal_waktu_valid.append((id,hari,bulan,waktu))
    else :
        data_tanggal_waktu_valid.append((id,hari,bulan,waktu))
df_tanggal = pd.DataFrame(data_tanggal_waktu_valid, columns=['id','hari', 'bulan', 'waktu'])
df_tanggal

,id,hari,bulan,waktu
0,1,MINGGU,JANUARI,Pagi
1,2,MINGGU,JANUARI,Siang
2,3,MINGGU,JANUARI,Siang
3,4,SENIN,JANUARI,Dini Hari
4,5,SENIN,JANUARI,Pagi
...,...,...,...,...
706,707,KAMIS,JUNI,Siang
707,708,KAMIS,JUNI,Siang
708,709,JUMAT,JUNI,Pagi
709,710,JUMAT,JUNI,Pagi


In [30]:
## alamat T ##
df_alamat = df[['no.','alamat TKP']]
# Mengambil baris data berdasarkan ID yang ada dalam data_asli
baris_terfilter = df_alamat[df_alamat['no.'].isin(df_tanggal['id'])]
data_alamat_valid = []
# Fungsi untuk mengidentifikasi bentuk geometri berdasarkan pola yang muncul dalam teks
def identifikasi_bentuk_geometri(teks):
    # Gunakan ekspresi reguler untuk mencari pola yang sesuai
    if re.search(r'SIMPANG 4', teks):
        return 'simpang 4'
    elif re.search(r'PANDUGO', teks):
        return 'lurus'
    elif re.search(r'SIMPANG 3', teks):
        return 'simpang 3'
    elif re.search(r'SETELAH', teks):
        return 'lurus'
    elif re.search(r'SEBELAH', teks):
        return 'lurus'
    elif re.search(r'BUNDARAN', teks):
        return 'bundaran'
    elif re.search(r'JEMBATAN', teks):
        return 'jembatan'
    elif re.search(r'KERETA API', teks):
        return 'rel kereta api'
    else:
        return 'lurus'

for index, row in baris_terfilter.iterrows():
    alamat = row['alamat TKP']
    id = row['no.']
    bentuk_geometri = identifikasi_bentuk_geometri(alamat)
    data_alamat_valid.append((id, bentuk_geometri))
df_alamat = pd.DataFrame(data_alamat_valid, columns=['id', 'bentuk geometri'])
df_alamat

,id,bentuk geometri
0,1,lurus
1,2,lurus
2,3,simpang 4
3,4,lurus
4,5,lurus
...,...,...
706,707,simpang 4
707,708,simpang 3
708,709,lurus
709,710,lurus


In [31]:
## Pihak yg terlibat ##

df_terlibat = df[['no.','pihak yang terlibat']]
# Mengambil baris data berdasarkan ID yang ada dalam data_asli
baris_terfilter = df_terlibat[df_terlibat['no.'].isin(df_tanggal['id'])]
data_terlibat_valid = []

def identifikasi_pihak_terlibat(teks):
    # Gunakan ekspresi reguler untuk mencari pola yang sesuai
    #R2 X R2 X R2  
    if re.search(r'R2.*R2.*R2', teks):
        return 'R2 X R2 X R2'
    #R2 X R4 X R4
    elif re.search(r'(R2.*R4.*R4)|(R4.*R2.*R4)|(R4.*R4.*R2)', teks):
        return 'R2 X R4 X R4'
    #R2 X R2 X BUS
    elif re.search(r'(BUS.*R2.*R2)|(R2.*BUS.*R2)|(R2.*R2.*BUS)', teks):
        return 'R2 X R2 X BUS'
    #R4 X R4 X BUS
    elif re.search(r'(BUS.*R4.*R4)|(R4.*BUS.*R4)|(R4.*R4.*BUS)', teks):
        return 'R4 X R4 X BUS'
    elif re.search(r'(BUS.*R4.*PICK UP)|(R4.*BUS.*PICK UP)|(PICK UP.*R4.*BUS)|(PICK UP.*BUS.*R4)', teks):
        return 'R4 X R4 X BUS'
    elif re.search(r'(BUS.*R4.*PIK UP)|(R4.*BUS.*PIK UP)|(PIK UP.*R4.*BUS)|(PIK UP.*BUS.*R4)', teks):
        return 'R4 X R4 X BUS'
    #R4 X R2 X R2
    elif re.search(r'(R4.*R2.*R2)|(R2.*R4.*R2)|(R2.*R2.*R4)', teks):
        return 'R4 X R2 X R2'
    #R2 X R4 X TRUK
    elif re.search(r'(R4.*R2.*TRUK)|(R2.*R4.*TRUK)|(R2.*TRUK.*R4)|(R4.*TRUK.*R2)|(TRUK.*R2.*R4)|(TRUK.*R4.*R2)', teks):
        return 'R2 X R4 X TRUK'
    #R2 X R2
    elif re.search(r'R2.*R2', teks):
        return 'R2 X R2'
    #R2 X R4
    elif re.search(r'(R2.*R4)|(R4.*R2)', teks):
        return 'R2 X R4'
    elif re.search(r'(R2.*TAXI)|(TAXI.*R2)', teks):
        return 'R2 X R4'
    elif re.search(r'(R2.*AMBULANCE)|(AMBULANCE.*R2)', teks):
        return 'R2 X R4'
    elif re.search(r'(R2.*PIK UP)|(PIK UP.*R2)', teks):
        return 'R2 X R4'
    #R4 X R4
    elif re.search(r'(R4.*R4)|(R4.*R4)', teks):
        return 'R4 X R4'
    #R2 X FORKLIF
    elif re.search(r'(R2.*FORKLIP)|(FORKLIP.*R2)', teks):
        return 'R2 X FORKLIFT'
    #R2 X BENTOR
    elif re.search(r'(R2.*BENTOR)|(BENTOR.*R2)', teks):
        return 'R2 X BECAK MOTOR'
    #R2 X BECAK
    elif re.search(r'(R2.*BECAK)|(BECAK.*R2)', teks):
        return 'R2 X BECAK'
    #R2 X PEJALAN KAKI
    elif re.search(r'(R2.*PEJALAN KAKI)|(PEJALAN KAKI.*R2)', teks):
        return 'R2 X PEJALAN KAKI'
    elif re.search(r'(R2.*PENYEBRANG)|(PENYEBRANG.*R2)', teks):
        return 'R2 X PEJALAN KAKI'
    elif re.search(r'(R2.*PETUGAS)|(PETUGAS.*R2)', teks):
        return 'R2 X PEJALAN KAKI'
    elif re.search(r'(R2.*PENYENBRANG)|(PENYENBRANG.*R2)', teks):
        return 'R2 X PEJALAN KAKI'
    #R4 X PEJALAN KAKI
    elif re.search(r'(R4.*PEJALAN KAKI)|(PEJALAN KAKI.*R4)', teks):
        return 'R4 X PEJALAN KAKI'
    elif re.search(r'(R4.*PENYEBRANG)|(PENYEBRANG.*R4)', teks):
        return 'R4 X PEJALAN KAKI'
    elif re.search(r'(R4.*PETUGAS)|(PETUGAS.*R4)', teks):
        return 'R4 X PEJALAN KAKI'
    elif re.search(r'(PIK UP.*PENYEBRANG)|(PENYEBRANG.*PIK UP)', teks):
        return 'R4 X PEJALAN KAKI'
    #R2 X KERETA API
    elif re.search(r'(R2.*KERETA)|(KERETA.*R2)', teks):
        return 'R2 X KERETA API'
    #R2 X BUS
    elif re.search(r'(R2.*BUS)|(BUS.*R2)', teks):
        return 'R2 X BUS'
    #R2 X TRUK
    elif re.search(r'(R2.*TRUK)|(TRUK.*R2)', teks):
        return 'R2 X TRUK'
    elif re.search(r'(R2.*TRAILER)|(TRAILER.*R2)', teks):
        return 'R2 X TRUK'
    elif re.search(r'(R2.*TRONTON)|(TRONTON.*R2)', teks):
        return 'R2 X TRUK'
    #R4 X TRUK
    elif re.search(r'(R4.*TRUK)|(TRUK.*R4)', teks):
        return 'R4 X TRUK'
    elif re.search(r'(R4.*TRAILER)|(TRAILER.*R4)', teks):
        return 'R4 X TRUK'
    #R3 X TRUK
    elif re.search(r'(R3.*TRUK)|(TRUK.*R3)', teks):
        return 'R3 X TRUK'
    #TRUK X TRUK
    elif re.search(r'TRUK.*TRUK', teks):
        return 'TRUK X TRUK'
    elif re.search(r'(TRAILER.*TRUK)|(TRUK.*TRAILER)', teks):
        return 'TRUK X TRUK'
    #SEPEDA ANGIN X R4
    elif re.search(r'(SEPEDA ANGIN.*R4)|(R4.*SEPEDA ANGIN)', teks):
        return 'SEPEDA ANGIN X R4'
    elif re.search(r'(SEPEDA ANGIN.*SEDAN)|(SEDAN.*SEPEDA ANGIN)', teks):
        return 'SEPEDA ANGIN X R4'
    #SEPEDA ANGIN X R2
    elif re.search(r'(SEPEDA ANGIN.*R2)|(R2.*SEPEDA ANGIN)', teks):
        return 'SEPEDA ANGIN X R2'
    #R2 X R3
    elif re.search(r'(R2.*R3)|(R3.*R2)', teks):
        return 'R2 X R3'
    #R3 x R3
    elif re.search(r'R3.*R3', teks):
        return 'R3 X R3'
    #R2
    elif re.search(r'R2', teks):
        return 'R2'
    #R3
    elif re.search(r'R3', teks):
        return 'R3'
    #R4
    elif re.search(r'R4', teks):
        return 'R4'
    elif re.search(r'PIK UP', teks):
        return 'R4'
    else :
        return 'lainnya'
        
for index, row in baris_terfilter.iterrows():
    kendaraan = row['pihak yang terlibat']
    id = row['no.']
    pihak_terlibat = identifikasi_pihak_terlibat(kendaraan)
    data_terlibat_valid.append((id, pihak_terlibat))
    # print("id= ",id,"pihak terlibat= ",pihak_terlibat,"kendaraan= ",kendaraan)
df_terlibat = pd.DataFrame(data_terlibat_valid, columns=['id', 'pihak terlibat'])
df_terlibat

,id,pihak terlibat
0,1,R2 X R2
1,2,R2 X R2
2,3,R2 X R4
3,4,R2 X R2
4,5,R2 X R4
...,...,...
706,707,R2 X R2
707,708,R2 X R2
708,709,R2 X R2
709,710,SEPEDA ANGIN X R2


In [32]:
## Tingkat kecelakaan ##

df_korban = df[['no.','meninggal','luka berat','luka ringan']]
baris_terfilter = df_korban[df_korban['no.'].isin(df_tanggal['id'])]
baris_terfilter = baris_terfilter.fillna(0)
data_korban_valid = []
def identifikasi_korban(teks):
    if row['meninggal'] >= 1.0:
        return 'berat'
    elif row['luka berat'] >= 1.0:
        return 'sedang'
    elif row['luka ringan'] >= 1.0:
        return 'ringan'
    else :
        return 'lainnya'

for index, row in baris_terfilter.iterrows():
    id = row['no.'].astype(int)
    data_korban = row[['meninggal','luka berat','luka ringan']]
    tingkat_kecelakaan = identifikasi_korban(data_korban)
    data_korban_valid.append((id,tingkat_kecelakaan))
df_tingkat_kecelakaan = pd.DataFrame(data_korban_valid, columns=['id', 'tingkat kecelakaan'])
df_tingkat_kecelakaan

,id,tingkat kecelakaan
0,1,ringan
1,2,ringan
2,3,ringan
3,4,ringan
4,5,ringan
...,...,...
706,707,ringan
707,708,ringan
708,709,ringan
709,710,ringan


In [33]:
## Kerugian Materiil ##
df_kerugian = df[['no.','kerugian materiil']]
baris_terfilter = df_kerugian[df_kerugian['no.'].isin(df_tanggal['id'])]
data_kerugian_valid = []

def identifikasi_kerugian(teks):
    if kerugian < 1000000 :
        return 'ringan'
    elif kerugian >= 1000000 and kerugian < 5000000 :
        return 'sedang'
    elif kerugian >= 5000000 :
        return 'berat'

for index, row in baris_terfilter.iterrows():
    id = row['no.']
    kerugian = row['kerugian materiil']
    tingkat_kerugian = identifikasi_kerugian(kerugian)
    data_kerugian_valid.append((id,tingkat_kerugian))
    # print(id, kerugian, tingkat_kerugian)

df_tingkat_kerugian = pd.DataFrame(data_kerugian_valid, columns=['id', 'tingkat kerugian'])
df_tingkat_kerugian

,id,tingkat kerugian
0,1,sedang
1,2,ringan
2,3,sedang
3,4,sedang
4,5,sedang
...,...,...
706,707,ringan
707,708,sedang
708,709,ringan
709,710,ringan


In [34]:
## Penggabungan Kolom Tabel ##
import pandas as pd

# Menggabungkan berdasarkan kolom 'id'
tabel_gabungan = pd.merge(df_tanggal,df_alamat, on='id')
tabel_gabungan = pd.merge(tabel_gabungan, df_terlibat, on='id')
tabel_gabungan = pd.merge(tabel_gabungan, df_tingkat_kecelakaan, on='id')
df_gabungan = pd.merge(tabel_gabungan, df_tingkat_kerugian, on='id')
df_gabungan

,id,hari,bulan,waktu,bentuk geometri,pihak terlibat,tingkat kecelakaan,tingkat kerugian
0,1,MINGGU,JANUARI,Pagi,lurus,R2 X R2,ringan,sedang
1,2,MINGGU,JANUARI,Siang,lurus,R2 X R2,ringan,ringan
2,3,MINGGU,JANUARI,Siang,simpang 4,R2 X R4,ringan,sedang
3,4,SENIN,JANUARI,Dini Hari,lurus,R2 X R2,ringan,sedang
4,5,SENIN,JANUARI,Pagi,lurus,R2 X R4,ringan,sedang
...,...,...,...,...,...,...,...,...
706,707,KAMIS,JUNI,Siang,simpang 4,R2 X R2,ringan,ringan
707,708,KAMIS,JUNI,Siang,simpang 3,R2 X R2,ringan,sedang
708,709,JUMAT,JUNI,Pagi,lurus,R2 X R2,ringan,ringan
709,710,JUMAT,JUNI,Pagi,lurus,SEPEDA ANGIN X R2,ringan,ringan


In [35]:
## Transformasi data ##

mapping_hari = {
    'MINGGU' : 'A1',
    'SENIN' : 'A2',
    'SELASA' : 'A3',
    'RABU' : 'A4',
    'KAMIS' : 'A5',
    'JUMAT' : 'A6',
    'SABTU' : 'A7'
}
mapping_bulan = {
    'JANUARI' : 'B1',
    'FEBRUARI' : 'B2',
    'MARET' : 'B3',
    'APRIL' : 'B4',
    'MEI' : 'B5',
    'JUNI' : 'B6',
    'JULI' : 'B7',
    'AGUSTUS' : 'B8',
    'SEPTEMBER' : 'B9',
    'OKTOBER' : 'B10',
    'NOVEMBER' : 'B11',
    'DESEMBER' : 'B12'
}
mapping_waktu = {
    'Dini Hari' : 'C1',
    'Pagi' : 'C2',
    'Siang' : 'C3',
    'Sore' : 'C4',
    'Petang' : 'C5',
    'Malam' : 'C6'
}
mapping_geometri = {
    'lurus' : 'D1',
    'bundaran' : 'D2',
    'simpang 3' : 'D3',
    'simpang 4' : 'D4',
    'jembatan' : 'D5',
    'rel kereta api' : 'D6'
}
mapping_terlibat = {
    'R2 X R2 X R2' : 'E1',
    'R2 X R4 X R4' : 'E2',
    'R2 X R2 X BUS' : 'E3',
    'R4 X R4 X BUS' : 'E4',
    'R4 X R2 X R2' : 'E5',
    'R2 X R4 X TRUK' : 'E6',
    'R2 X R2' : 'E7',
    'R2 X R4' : 'E8',
    'R4 X R4' : 'E9',
    'R2 X FORKLIFT' : 'E10',
    'R2 X BECAK MOTOR' : 'E11',
    'R2 X PEJALAN KAKI' : 'E12',
    'R4 X PEJALAN KAKI' : 'E13',
    'R2 X KERETA API' : 'E14',
    'R2 X BUS' : 'E15',
    'R2 X TRUK' : 'E16',
    'R4 X TRUK' : 'E17',
    'R3 X TRUK' : 'E18',
    'TRUK X TRUK' : 'E19',
    'SEPEDA ANGIN X R4' : 'E20',
    'SEPEDA ANGIN X R2' : 'E21',
    'R2 X R3' : 'E22',
    'R3 X R3' : 'E23',
    'R2' : 'E24',
    'R3' : 'E25',
    'R4' : 'E26',
    'R2 X BECAK' : 'E27'
}
mapping_tingkat_kecelakaan = {
    'ringan' : 'F1',
    'sedang' : 'F2',
    'berat' : 'F3'
}
mapping_tingkat_kerugian = {
    'ringan' : 'G1',
    'sedang' : 'G2',
    'berat' : 'G3'
}
df_transform = df_gabungan.replace({'hari': mapping_hari, 'bulan': mapping_bulan, 'waktu': mapping_waktu, 
                                        'bentuk geometri': mapping_geometri, 'pihak terlibat': mapping_terlibat,
                                        'tingkat kecelakaan': mapping_tingkat_kecelakaan, 'tingkat kerugian': mapping_tingkat_kerugian})

df_transform

,id,hari,bulan,waktu,bentuk geometri,pihak terlibat,tingkat kecelakaan,tingkat kerugian
0,1,A1,B1,C2,D1,E7,F1,G2
1,2,A1,B1,C3,D1,E7,F1,G1
2,3,A1,B1,C3,D4,E8,F1,G2
3,4,A2,B1,C1,D1,E7,F1,G2
4,5,A2,B1,C2,D1,E8,F1,G2
...,...,...,...,...,...,...,...,...
706,707,A5,B6,C3,D4,E7,F1,G1
707,708,A5,B6,C3,D3,E7,F1,G2
708,709,A6,B6,C2,D1,E7,F1,G1
709,710,A6,B6,C2,D1,E21,F1,G1


In [36]:
df_transform['items'] = df_transform[['hari', 'bulan', 'waktu', 'bentuk geometri', 'pihak terlibat', 'tingkat kecelakaan', 'tingkat kerugian']].apply(lambda x: ' '.join(x), axis=1)
df_data = df_transform.drop(columns=['hari', 'bulan', 'waktu', 'bentuk geometri', 'pihak terlibat', 'tingkat kecelakaan', 'tingkat kerugian'])
df_data

,id,items
0,1,A1 B1 C2 D1 E7 F1 G2
1,2,A1 B1 C3 D1 E7 F1 G1
2,3,A1 B1 C3 D4 E8 F1 G2
3,4,A2 B1 C1 D1 E7 F1 G2
4,5,A2 B1 C2 D1 E8 F1 G2
...,...,...
706,707,A5 B6 C3 D4 E7 F1 G1
707,708,A5 B6 C3 D3 E7 F1 G2
708,709,A6 B6 C2 D1 E7 F1 G1
709,710,A6 B6 C2 D1 E21 F1 G1


In [37]:
## Proses algoritma apriori ##

In [38]:
data=df_data.drop(['id'],axis=1)

In [39]:
# Membuat list dalam list dari transaksi
records = []
for i in range(data.shape[0]):
    records.append([str(data.values[i,j]).split(' ') for j in range(data.shape[1])])

dataset = [[] for trx in range(len(records))]
for i in range(len(records)):
    for j in records[i][0]:
        dataset[i].append(j)
dataset

[['A1', 'B1', 'C2', 'D1', 'E7', 'F1', 'G2'],
 ['A1', 'B1', 'C3', 'D1', 'E7', 'F1', 'G1'],
 ['A1', 'B1', 'C3', 'D4', 'E8', 'F1', 'G2'],
 ['A2', 'B1', 'C1', 'D1', 'E7', 'F1', 'G2'],
 ['A2', 'B1', 'C2', 'D1', 'E8', 'F1', 'G2'],
 ['A3', 'B1', 'C2', 'D1', 'E8', 'F1', 'G2'],
 ['A3', 'B1', 'C3', 'D1', 'E12', 'F1', 'G1'],
 ['A3', 'B1', 'C4', 'D1', 'E7', 'F1', 'G1'],
 ['A3', 'B1', 'C5', 'D1', 'E7', 'F1', 'G2'],
 ['A3', 'B1', 'C6', 'D1', 'E8', 'F1', 'G2'],
 ['A4', 'B1', 'C2', 'D4', 'E12', 'F3', 'G1'],
 ['A4', 'B1', 'C2', 'D6', 'E14', 'F3', 'G1'],
 ['A4', 'B1', 'C3', 'D1', 'E7', 'F1', 'G1'],
 ['A4', 'B1', 'C4', 'D1', 'E7', 'F1', 'G1'],
 ['A5', 'B1', 'C2', 'D1', 'E7', 'F1', 'G1'],
 ['A5', 'B1', 'C4', 'D1', 'E7', 'F1', 'G1'],
 ['A5', 'B1', 'C4', 'D1', 'E2', 'F1', 'G2'],
 ['A5', 'B1', 'C6', 'D1', 'E8', 'F1', 'G3'],
 ['A5', 'B1', 'C6', 'D1', 'E17', 'F1', 'G3'],
 ['A6', 'B1', 'C3', 'D1', 'E7', 'F1', 'G2'],
 ['A6', 'B1', 'C4', 'D1', 'E12', 'F1', 'G1'],
 ['A6', 'B1', 'C4', 'D1', 'E7', 'F3', 'G2'],
 ['A6

In [40]:
te = TransactionEncoder()
te_try = te.fit(dataset).transform(dataset)

In [41]:
df = pd.DataFrame(te_try, columns=te.columns_)
df

,A1,A2,A3,A4,A5,A6,A7,B1,B2,B3,...,E6,E7,E8,E9,F1,F2,F3,G1,G2,G3
0,True,False,False,False,False,False,False,True,False,False,...,False,True,False,False,True,False,False,False,True,False
1,True,False,False,False,False,False,False,True,False,False,...,False,True,False,False,True,False,False,True,False,False
2,True,False,False,False,False,False,False,True,False,False,...,False,False,True,False,True,False,False,False,True,False
3,False,True,False,False,False,False,False,True,False,False,...,False,True,False,False,True,False,False,False,True,False
4,False,True,False,False,False,False,False,True,False,False,...,False,False,True,False,True,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
706,False,False,False,False,True,False,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
707,False,False,False,False,True,False,False,False,False,False,...,False,True,False,False,True,False,False,False,True,False
708,False,False,False,False,False,True,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
709,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,True,False,False,True,False,False


In [42]:
# Build up the frequent items 
import time
%pip install memory_profiler
from memory_profiler import memory_usage

def run_apriori(df):
    start_time = time.time()
    frequent_itemsets_apriori = apriori(df, min_support=0.25, use_colnames=True)
    end_time = time.time()
    apriori_time = end_time - start_time
    return frequent_itemsets_apriori, apriori_time

# Mengukur penggunaan memori dan waktu eksekusi untuk Apriori
apriori_memory_usage = memory_usage((run_apriori, (df,)))
frequent_itemsets_apriori, apriori_time = run_apriori(df)

# Menampilkan hasil
print(f"Apriori time: {apriori_time:.4f} seconds, memory usage: {max(apriori_memory_usage):.2f} MiB")
frequent_itemsets_apriori.head()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Apriori time: 0.0030 seconds, memory usage: 48.14 MiB


,support,itemsets
0,0.275668,(C2)
1,0.263010,(C3)
2,0.859353,(D1)
3,0.464135,(E7)
4,0.257384,(E8)


In [43]:
# Create the apriori rules
frequent_itemsets_apriori = apriori(df, min_support=0.08, use_colnames=True)
rules_apriori = association_rules(frequent_itemsets_apriori, metric="lift", min_threshold=1)
rules_apriori

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(D1),(A1),0.859353,0.137834,0.123769,0.144026,1.044925,0.005321,1.007234,0.305682
1,(A1),(D1),0.137834,0.859353,0.123769,0.897959,1.044925,0.005321,1.378340,0.049867
2,(A1),(F1),0.137834,0.855134,0.125176,0.908163,1.062013,0.007309,1.577434,0.067727
3,(F1),(A1),0.855134,0.137834,0.125176,0.146382,1.062013,0.007309,1.010013,0.403076
4,(A3),(D1),0.130802,0.859353,0.119550,0.913978,1.063566,0.007145,1.635021,0.068761
...,...,...,...,...,...,...,...,...,...,...
369,"(E8, D1)","(G2, F1)",0.218003,0.382560,0.116737,0.535484,1.399739,0.033338,1.329212,0.365195
370,"(G2, F1)","(E8, D1)",0.382560,0.218003,0.116737,0.305147,1.399739,0.033338,1.125414,0.462524
371,"(D1, G2)","(E8, F1)",0.381153,0.215190,0.116737,0.306273,1.423269,0.034717,1.131295,0.480559
372,(E8),"(D1, G2, F1)",0.257384,0.316456,0.116737,0.453552,1.433224,0.035286,1.250886,0.407037


In [44]:
## FP-GROWTH RULES ##

In [45]:
#Importing Libraries
from mlxtend.frequent_patterns import fpgrowth
#running the fpgrowth algorithm
def run_fpgrowth(df):
    start_time = time.time()
    frequent_itemsets_fpgrowth = fpgrowth(df, min_support=0.25, use_colnames=True)
    end_time = time.time()
    fpgrowth_time = end_time - start_time
    return frequent_itemsets_fpgrowth, fpgrowth_time

# Mengukur penggunaan memori dan waktu eksekusi untuk FP-Growth
fpgrowth_memory_usage = memory_usage((run_fpgrowth, (df,)))
frequent_itemsets_fpgrowth, fpgrowth_time = run_fpgrowth(df)
print(f"FP-Growth time: {fpgrowth_time:.4f} seconds, memory usage: {max(fpgrowth_memory_usage):.2f} MiB")
frequent_itemsets_fpgrowth.head()

FP-Growth time: 0.0209 seconds, memory usage: 50.07 MiB


,support,itemsets
0,0.859353,(D1)
1,0.855134,(F1)
2,0.464135,(E7)
3,0.462729,(G2)
4,0.275668,(C2)


In [46]:
# creating asssociation rules
frequent_itemsets_fpgrowth = fpgrowth(df, min_support=0.08, use_colnames=True)
rules_fpgrowth=association_rules(frequent_itemsets_fpgrowth, metric="lift", min_threshold=1)
# printing association rules
rules_fpgrowth.sort_values('confidence',ascending=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
212,"(D1, E12)",(G1),0.094233,0.517581,0.094233,1.000000,1.932065,0.045460,inf,0.532609
208,(E12),(G1),0.101266,0.517581,0.101266,1.000000,1.932065,0.048853,inf,0.536776
218,"(E12, F1)",(G1),0.080169,0.517581,0.080169,1.000000,1.932065,0.038675,inf,0.524465
18,"(E7, G1, D1)",(F1),0.205345,0.855134,0.201125,0.979452,1.145379,0.025528,7.050164,0.159725
10,"(E7, G1)",(F1),0.240506,0.855134,0.234880,0.976608,1.142053,0.029215,6.193038,0.163772
...,...,...,...,...,...,...,...,...,...,...
371,(D1),"(B6, G1)",0.859353,0.092827,0.084388,0.098200,1.057878,0.004617,1.005958,0.389000
349,(F1),"(B5, G1)",0.855134,0.092827,0.082982,0.097039,1.045380,0.003602,1.004665,0.299654
329,(D1),"(E7, B4)",0.859353,0.091421,0.082982,0.096563,1.056251,0.004419,1.005692,0.378644
347,(F1),"(E7, B5)",0.855134,0.088608,0.081575,0.095395,1.076598,0.005804,1.007503,0.491128


In [47]:
## Deploy bentu pickle ##

In [48]:
with open('processed_results.pkl', 'wb') as file:
    pickle.dump(rules_apriori, file)
    pickle.dump(rules_fpgrowth, file)
    pickle.dump(df_gabungan, file)

In [49]:
from mlxtend.frequent_patterns import apriori

%timeit -n 100 -r 10 apriori(df, min_support=0.6)

7.07 ms ± 1.33 ms per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [50]:
from mlxtend.frequent_patterns import fpgrowth

%timeit -n 100 -r 10 fpgrowth(df, min_support=0.6)

6.3 ms ± 2.63 ms per loop (mean ± std. dev. of 10 runs, 100 loops each)
